In [1]:
!pip -q install -U transformers datasets accelerate sentencepiece scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.5 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [3]:
from google.colab import files

uploaded = files.upload()

Saving CTS_CRM_15000_NLP_Preprocessed.csv to CTS_CRM_15000_NLP_Preprocessed.csv


In [4]:
from google.colab import files

uploaded = files.upload()

Saving 02_hidden_llm_benchmark_15000.csv to 02_hidden_llm_benchmark_15000.csv


In [5]:
#Load the datasets
BENCHMARK_PATH = "/content/02_hidden_llm_benchmark_15000.csv"
CRM_PATH = "/content/CTS_CRM_15000_NLP_Preprocessed.csv"

benchmark = pd.read_csv(BENCHMARK_PATH)
crm = pd.read_csv(CRM_PATH)

print("Benchmark shape:", benchmark.shape)
print("CRM shape:", crm.shape)

print("\nBenchmark columns:")
print(benchmark.columns.tolist())

print("\nCRM columns:")
print(crm.columns.tolist())

Benchmark shape: (15000, 10)
CRM shape: (15000, 23)

Benchmark columns:
['interaction_id', 'crm_note_id', 'reference_sentiment', 'reference_topics', 'reference_objections', 'reference_primary_objection', 'reference_objection_severity', 'reference_aspect_sentiments', 'split_assignment', 'reference_source']

CRM columns:
['interaction_id', 'crm_note_id', 'interaction_date', 'rep_id', 'hcp_id', 'city', 'region', 'territory_id', 'hcp_specialization', 'therapeutic_area', 'drug_id', 'drug_name', 'brand_name', 'crm_note', 'word_count_raw', 'text_raw', 'text_lower', 'text_normalized', 'text_no_admin', 'text_no_punct', 'text_no_stopwords', 'clean_text', 'clean_word_count']


In [6]:
#Merge CRM text with benchmark labels
label_cols = [
    "interaction_id",
    "reference_primary_objection",
    "split_assignment"
]

data = crm.merge(
    benchmark[label_cols],
    on="interaction_id",
    how="inner"
)

print("Merged shape:", data.shape)

print("\nSplit counts:")
print(data["split_assignment"].value_counts())

print("\nSample:")
display(
    data[
        [
            "interaction_id",
            "crm_note",
            "reference_primary_objection",
            "split_assignment"
        ]
    ].head()
)

Merged shape: (15000, 25)

Split counts:
split_assignment
train             10546
validation         1521
phrase_holdout     1471
final_hidden       1462
Name: count, dtype: int64

Sample:


,interaction_id,crm_note,reference_primary_objection,split_assignment
0,INT000001,Saw HCP re Arthrelis. 2 patients mentioned inj...,Side Effects / Tolerability,train
1,INT000002,Met on Arthrelis. HCP reports fewer tolerance ...,Adherence Concern,train
2,INT000003,Met on Arthrelis. the previous adherence conce...,Prior Authorization,train
3,INT000004,The HCP reviewed recent experience with Arthre...,High Out-of-Pocket Cost,train
4,INT000005,Brief discussion focused on Rheumora. payer ap...,Prior Authorization,train


In [7]:
#Preprocess the CRM text
def normalize_crm_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    replacements = [
        (r"\bf\s*/\s*u\b", "follow up"),
        (r"\bpa\b", "prior authorization"),
        (r"\boop\b", "out of pocket"),
        (r"\bhcp\b", "healthcare professional"),
        (r"\bpts\b", "patients"),
        (r"\bpt\b", "patient"),
        (r"\brx\b", "prescription"),
    ]

    for pattern, replacement in replacements:
        text = re.sub(
            pattern,
            replacement,
            text,
            flags=re.IGNORECASE
        )

    text = re.sub(
        r"\bREP\d+\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


data["model_text"] = (
    data["crm_note"]
    .fillna("")
    .apply(normalize_crm_text)
)

display(
    data[
        ["crm_note", "model_text"]
    ].head(10)
)

,crm_note,model_text
0,Saw HCP re Arthrelis. 2 patients mentioned inj...,Saw healthcare professional re Arthrelis. 2 pa...
1,Met on Arthrelis. HCP reports fewer tolerance ...,Met on Arthrelis. healthcare professional repo...
2,Met on Arthrelis. the previous adherence conce...,Met on Arthrelis. the previous adherence conce...
3,The HCP reviewed recent experience with Arthre...,The healthcare professional reviewed recent ex...
4,Brief discussion focused on Rheumora. payer ap...,Brief discussion focused on Rheumora. payer ap...
5,Quick f/u on Arthrelis. office reports fewer p...,Quick follow up on Arthrelis. office reports f...
6,Arthrelis discussion. use in routine practice ...,Arthrelis discussion. use in routine practice ...
7,Saw HCP re Arthrelis. HCP remains cautious abo...,Saw healthcare professional re Arthrelis. heal...
8,Rheumora discussion. recent approvals have gon...,Rheumora discussion. recent approvals have gon...
9,Saw HCP re Renovia. practice wants a simpler d...,Saw healthcare professional re Renovia. practi...


In [8]:
#Create the four splits
train_data = data[
    data["split_assignment"] == "train"
].copy()

val_data = data[
    data["split_assignment"] == "validation"
].copy()

phrase_data = data[
    data["split_assignment"] == "phrase_holdout"
].copy()

hidden_data = data[
    data["split_assignment"] == "final_hidden"
].copy()

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Phrase holdout:", len(phrase_data))
print("Final hidden:", len(hidden_data))

Train: 10546
Validation: 1521
Phrase holdout: 1471
Final hidden: 1462


In [9]:
#Encode the objection labels
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(
    train_data["reference_primary_objection"].astype(str)
)

y_val = label_encoder.transform(
    val_data["reference_primary_objection"].astype(str)
)

y_phrase = label_encoder.transform(
    phrase_data["reference_primary_objection"].astype(str)
)

y_hidden = label_encoder.transform(
    hidden_data["reference_primary_objection"].astype(str)
)

OBJECTION_CLASSES = list(
    label_encoder.classes_
)

NUM_LABELS = len(OBJECTION_CLASSES)

print("Number of objection classes:", NUM_LABELS)

for i, label in enumerate(OBJECTION_CLASSES):
    print(i, "→", label)

Number of objection classes: 13
0 → Adherence Concern
1 → Clinical Evidence Gap
2 → Competitor Preference
3 → Dosing Complexity
4 → Efficacy Concern
5 → Eligibility Criteria
6 → Formulary Restriction
7 → High Out-of-Pocket Cost
8 → No Objection
9 → Poor Availability
10 → Prior Authorization
11 → Safety Concern
12 → Side Effects / Tolerability


In [10]:
#Check class distribution
class_counts = (
    train_data["reference_primary_objection"]
    .value_counts()
    .reindex(OBJECTION_CLASSES)
    .fillna(0)
)

print(class_counts)

print("\nSmallest class:", class_counts.min())
print("Largest class:", class_counts.max())

reference_primary_objection
Adherence Concern               614
Clinical Evidence Gap           676
Competitor Preference           609
Dosing Complexity               430
Efficacy Concern                685
Eligibility Criteria            435
Formulary Restriction           526
High Out-of-Pocket Cost         873
No Objection                   2559
Poor Availability               620
Prior Authorization            1042
Safety Concern                  642
Side Effects / Tolerability     835
Name: count, dtype: int64

Smallest class: 430
Largest class: 2559


In [11]:
#Load DistilBERT tokenizer
MODEL_NAME = "distilbert-base-uncased"

MAX_LENGTH = 256
LEARNING_RATE = 2e-5
MAX_EPOCHS = 5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Model: distilbert-base-uncased
Max length: 256


In [12]:
#Create Hugging Face datasets
train_frame = pd.DataFrame({
    "text": train_data["model_text"].tolist(),
    "labels": y_train.tolist()
})

val_frame = pd.DataFrame({
    "text": val_data["model_text"].tolist(),
    "labels": y_val.tolist()
})

phrase_frame = pd.DataFrame({
    "text": phrase_data["model_text"].tolist(),
    "labels": y_phrase.tolist()
})

hidden_frame = pd.DataFrame({
    "text": hidden_data["model_text"].tolist(),
    "labels": y_hidden.tolist()
})

In [13]:
train_ds = Dataset.from_pandas(
    train_frame,
    preserve_index=False
)

val_ds = Dataset.from_pandas(
    val_frame,
    preserve_index=False
)

phrase_ds = Dataset.from_pandas(
    phrase_frame,
    preserve_index=False
)

hidden_ds = Dataset.from_pandas(
    hidden_frame,
    preserve_index=False
)

In [14]:
#Tokenize
def tokenize_batch(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

In [15]:
train_ds = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

val_ds = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

phrase_ds = phrase_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

hidden_ds = hidden_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/10546 [00:00<?, ? examples/s]

Map:   0%|          | 0/1521 [00:00<?, ? examples/s]

Map:   0%|          | 0/1471 [00:00<?, ? examples/s]

Map:   0%|          | 0/1462 [00:00<?, ? examples/s]

In [16]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print(train_ds)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10546
})


In [17]:
#Create class weights
counts = np.array(
    [
        class_counts[label]
        for label in OBJECTION_CLASSES
    ],
    dtype=np.float32
)

total = counts.sum()

class_weights = total / (
    NUM_LABELS *
    np.maximum(counts, 1.0)
)

class_weights = (
    class_weights /
    class_weights.mean()
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

for label, weight in zip(
    OBJECTION_CLASSES,
    class_weights.numpy()
):
    print(
        f"{label}: {weight:.3f}"
    )

Adherence Concern: 1.080
Clinical Evidence Gap: 0.981
Competitor Preference: 1.089
Dosing Complexity: 1.543
Efficacy Concern: 0.968
Eligibility Criteria: 1.525
Formulary Restriction: 1.261
High Out-of-Pocket Cost: 0.760
No Objection: 0.259
Poor Availability: 1.070
Prior Authorization: 0.637
Safety Concern: 1.033
Side Effects / Tolerability: 0.794


In [18]:
#Load DistilBERT
id2label = {
    i: label
    for i, label in enumerate(
        OBJECTION_CLASSES
    )
}

label2id = {
    label: i
    for i, label in enumerate(
        OBJECTION_CLASSES
    )
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

print("DistilBERT loaded.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT loaded.


In [19]:
#Define Accuracy + Macro-F1
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),

        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    }

In [20]:
#Weighted Trainer
class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        weights = class_weights.to(
            logits.device
        )

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=weights
        )

        loss = loss_fn(
            logits.view(-1, NUM_LABELS),
            labels.view(-1)
        )

        if return_outputs:
            return loss, outputs

        return loss

In [58]:
OUTPUT_DIR = "/content/final_distilbert_objection_model"

# Calculate total training steps and warmup steps
total_train_steps = int((len(train_ds) / 8 / 2) * MAX_EPOCHS)
warmup_steps = int(total_train_steps * WARMUP_RATIO)

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    gradient_accumulation_steps=2,

    num_train_epochs=MAX_EPOCHS,

    weight_decay=WEIGHT_DECAY,

    # Replaced warmup_ratio with warmup_steps
    warmup_steps=warmup_steps,

    lr_scheduler_type="cosine",

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,

    save_total_limit=2,

    logging_strategy="steps",
    logging_steps=50,

    # Disabling fp16 for troubleshooting
    fp16=False,

    report_to="none",

    seed=42
)

In [25]:
OUTPUT_DIR = "/content/final_distilbert_objection_model"

# Calculate total training steps and warmup steps
total_train_steps = int((len(train_ds) / 8 / 2) * MAX_EPOCHS)
warmup_steps = int(total_train_steps * WARMUP_RATIO)

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    gradient_accumulation_steps=2,

    num_train_epochs=MAX_EPOCHS,

    weight_decay=WEIGHT_DECAY,

    # Replaced warmup_ratio with warmup_steps
    warmup_steps=warmup_steps,

    lr_scheduler_type="cosine",

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,

    save_total_limit=2,

    logging_strategy="steps",
    logging_steps=50,

    # Disabling fp16 for troubleshooting, as the error message is truncated.
    fp16=False,

    report_to="none",

    seed=42
)

In [26]:
#Start training
trainer = WeightedTrainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=val_ds,

    # Removed the tokenizer argument as it's not directly supported by Trainer
    # tokenizer=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.103989,0.234121,0.969099,0.961877
2,0.051695,0.212971,0.972387,0.966085
3,0.007757,0.276508,0.969757,0.961746
4,0.043425,0.265255,0.966469,0.957741


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2640, training_loss=0.07388367328905698, metrics={'train_runtime': 318.8326, 'train_samples_per_second': 165.385, 'train_steps_per_second': 10.35, 'total_flos': 651300215440416.0, 'train_loss': 0.07388367328905698, 'epoch': 4.0})

In [27]:
#Check validation performance
val_metrics = trainer.evaluate(
    eval_dataset=val_ds
)

print("FINAL VALIDATION RESULTS")

print(
    "Accuracy:",
    round(
        val_metrics["eval_accuracy"],
        4
    )
)

print(
    "Macro-F1:",
    round(
        val_metrics["eval_macro_f1"],
        4
    )
)

print(
    "\nBest checkpoint:",
    trainer.state.best_model_checkpoint
)


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.043425,0.212971,4,0.972387,0.966085


FINAL VALIDATION RESULTS
Accuracy: 0.9724
Macro-F1: 0.9661

Best checkpoint: /content/final_distilbert_objection_model/checkpoint-1320


In [28]:
#Test phrase holdout
phrase_metrics = trainer.evaluate(
    eval_dataset=phrase_ds,
    metric_key_prefix="phrase"
)

print("PHRASE HOLDOUT")

print(
    "Accuracy:",
    round(
        phrase_metrics["phrase_accuracy"],
        4
    )
)

print(
    "Macro-F1:",
    round(
        phrase_metrics["phrase_macro_f1"],
        4
    )
)

[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.043425,0.274729,4,0.958532,0.949408


PHRASE HOLDOUT
Accuracy: 0.9585
Macro-F1: 0.9494


In [29]:
#Final hidden evaluation
hidden_metrics = trainer.evaluate(
    eval_dataset=hidden_ds,
    metric_key_prefix="hidden"
)

print("FINAL HIDDEN RESULTS")

print(
    "Accuracy:",
    round(
        hidden_metrics["hidden_accuracy"],
        4
    )
)

print(
    "Macro-F1:",
    round(
        hidden_metrics["hidden_macro_f1"],
        4
    )
)

[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.043425,0.399944,4,0.951436,0.942205


FINAL HIDDEN RESULTS
Accuracy: 0.9514
Macro-F1: 0.9422


In [30]:
#Save your final model
FINAL_MODEL_DIR = (
    "/content/final_distilbert_objection_model"
)

trainer.save_model(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)

with open(
    os.path.join(
        FINAL_MODEL_DIR,
        "label_mapping.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "id2label": id2label,
            "label2id": label2id
        },
        f,
        indent=2
    )

print(
    "Model saved to:",
    FINAL_MODEL_DIR
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/final_distilbert_objection_model


In [43]:
#Test your NEW unseen CSV
from google.colab import files

uploaded = files.upload()

unseen_filename = list(
    uploaded.keys()
)[0]

unseen_df = pd.read_csv(
    f"/content/{unseen_filename}"
)

print(
    "Unseen shape:",
    unseen_df.shape
)

display(
    unseen_df.head()
)

Saving final_unseen_objection_test_260.csv to final_unseen_objection_test_260.csv
Unseen shape: (260, 3)


,test_id,crm_note,ground_truth
0,UNSEEN_0031,The HCP is concerned that the patient's out-of...,High Out-of-Pocket Cost
1,UNSEEN_0072,The practice noted that the physician asked ho...,Efficacy Concern
2,UNSEEN_0220,The HCP explained that the physician is concer...,Dosing Complexity
3,UNSEEN_0185,The HCP explained that the HCP is concerned th...,Formulary Restriction
4,UNSEEN_0201,The physician is concerned that the dosing ins...,Dosing Complexity


In [44]:
#Apply the SAME preprocessing
unseen_df["model_text"] = (
    unseen_df["crm_note"]
    .fillna("")
    .apply(normalize_crm_text)
)

In [45]:
unseen_frame = pd.DataFrame({
    "text": unseen_df["model_text"].tolist()
})

unseen_ds = Dataset.from_pandas(
    unseen_frame,
    preserve_index=False
)

unseen_ds = unseen_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/260 [00:00<?, ? examples/s]

In [46]:
#Predict objections
unseen_output = trainer.predict(
    unseen_ds
)

unseen_pred_ids = np.argmax(
    unseen_output.predictions,
    axis=-1
)

unseen_pred_labels = (
    label_encoder.inverse_transform(
        unseen_pred_ids
    )
)

unseen_df["predicted_objection"] = (
    unseen_pred_labels
)

display(
    unseen_df[
        [
            "crm_note",
            "predicted_objection"
        ]
    ]
)

,crm_note,predicted_objection
0,The HCP is concerned that the patient's out-of...,High Out-of-Pocket Cost
1,The practice noted that the physician asked ho...,Efficacy Concern
2,The HCP explained that the physician is concer...,Dosing Complexity
3,The HCP explained that the HCP is concerned th...,Formulary Restriction
4,The physician is concerned that the dosing ins...,Dosing Complexity
...,...,...
255,The physician mentioned that the clinic is con...,Side Effects / Tolerability
256,"In a follow-up interaction, the physician want...",Safety Concern
257,"In a follow-up interaction, the HCP reports th...",Prior Authorization
258,The physician requested guidance on eligibilit...,Eligibility Criteria


In [47]:
# Predict on unseen dataset
unseen_output = trainer.predict(unseen_ds)

pred_ids = np.argmax(
    unseen_output.predictions,
    axis=-1
)

predicted_labels = label_encoder.inverse_transform(pred_ids)

unseen_df["predicted_objection"] = predicted_labels

In [49]:
from sklearn.metrics import accuracy_score, f1_score

y_true = unseen_df["ground_truth"]
y_pred = unseen_df["predicted_objection"]

accuracy = accuracy_score(y_true, y_pred)

f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

print("UNSEEN DATA RESULTS")
print("-------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Macro F1 : {f1:.4f}")
print(f"Macro F1 : {f1 * 100:.2f}%")

UNSEEN DATA RESULTS
-------------------
Accuracy : 0.7500
Accuracy : 75.00%
Macro F1 : 0.7302
Macro F1 : 73.02%


In [52]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        zero_division=0
    )
)

                             precision    recall  f1-score   support

          Adherence Concern       0.73      0.55      0.63        20
      Clinical Evidence Gap       0.64      0.90      0.75        20
      Competitor Preference       1.00      0.75      0.86        20
          Dosing Complexity       0.57      1.00      0.73        20
           Efficacy Concern       0.61      1.00      0.75        20
       Eligibility Criteria       0.80      1.00      0.89        20
      Formulary Restriction       1.00      1.00      1.00        20
    High Out-of-Pocket Cost       1.00      0.75      0.86        20
               No Objection       0.40      0.10      0.16        20
          Poor Availability       1.00      0.35      0.52        20
        Prior Authorization       1.00      1.00      1.00        20
             Safety Concern       1.00      0.60      0.75        20
Side Effects / Tolerability       0.50      0.75      0.60        20

                   accuracy     

In [53]:
print("MODEL LABELS")
print("=" * 40)

for i, label in enumerate(label_encoder.classes_):
    print(i, "->", label)

print("\nUNSEEN LABELS")
print("=" * 40)

for label in sorted(unseen_df["ground_truth"].unique()):
    print("->", label)

MODEL LABELS
0 -> Adherence Concern
1 -> Clinical Evidence Gap
2 -> Competitor Preference
3 -> Dosing Complexity
4 -> Efficacy Concern
5 -> Eligibility Criteria
6 -> Formulary Restriction
7 -> High Out-of-Pocket Cost
8 -> No Objection
9 -> Poor Availability
10 -> Prior Authorization
11 -> Safety Concern
12 -> Side Effects / Tolerability

UNSEEN LABELS
-> Adherence Concern
-> Clinical Evidence Gap
-> Competitor Preference
-> Dosing Complexity
-> Efficacy Concern
-> Eligibility Criteria
-> Formulary Restriction
-> High Out-of-Pocket Cost
-> No Objection
-> Poor Availability
-> Prior Authorization
-> Safety Concern
-> Side Effects / Tolerability


In [55]:
print("Number of model classes:",
      len(label_encoder.classes_))

print("Number of unseen classes:",
      unseen_df["ground_truth"].nunique())

Number of model classes: 13
Number of unseen classes: 13
